# Minimal RAG on your Elastic docs clone

**What you will do:** turn a *small* folder of Markdown into searchable chunks, run a question, and **read the retrieved passages** before any “AI answer.” That is the core of RAG.

**What you need:** Python 3.10+ on your Mac, this repo cloned, and about 15 minutes the first time (downloads embedding model once).

**Optional:** [Ollama](https://ollama.com) for a **local** answer step. If you skip it, you still learn retrieval—and you can paste the generated prompt into any chat tool you already use.

**License note:** This repository is CC BY-NC-ND. Use this notebook for **personal learning** only; do not republish substantial excerpts of the docs outside what the license allows.

## 1. Install packages (run once per environment)

From Terminal, many people prefer a virtual environment:

```bash
cd /path/to/docs-content/learning-rag
python3 -m venv .venv
source .venv/bin/activate
pip install -U pip chromadb sentence-transformers tqdm
```

Or run the next cell if you use Jupyter’s built-in install.

In [1]:
%pip install -q chromadb sentence-transformers tqdm

You should consider upgrading via the '/Users/mohan/repos/docs-content/learning-rag/.venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Choose which docs to index

Set **`CORPUS_FOLDER`** to a path under the repo (relative to the clone root). Examples:

- `"get-started"` — fundamentals (smaller; good first run)
- `"solutions"` — search, observability, and security use cases (~600+ files; raise `MAX_FILES` or start with a subfolder like `"solutions/search/get-started"`)

Each corpus gets its **own Chroma collection** on disk, so switching folders does not overwrite a previous index.

The source files use placeholders like `{{es}}` in the Markdown. You may see those in snippets—that is normal for this corpus.

In [5]:
from pathlib import Path


def find_repo_root() -> Path:
    """Walk upward from cwd until this clone looks like docs-content."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "get-started").is_dir() and (p / "README.md").is_file():
            return p
    return (here / "..").resolve()


REPO_ROOT = find_repo_root()

# --- change these two lines to switch corpora ---
CORPUS_FOLDER = "solutions"  # e.g. "get-started", "solutions", "solutions/observability"
MAX_FILES = 100  # solutions has 600+ files; start bounded, raise later

CORPUS_SUBDIR = (REPO_ROOT / CORPUS_FOLDER).resolve()
COLLECTION_NAME = "elastic_" + CORPUS_FOLDER.replace("/", "_").replace("-", "_")

assert CORPUS_SUBDIR.is_dir(), f"Missing folder: {CORPUS_SUBDIR}"
print("Indexing under:", CORPUS_SUBDIR)
print("Chroma collection:", COLLECTION_NAME)

Indexing under: /Users/mohan/repos/docs-content/solutions
Chroma collection: elastic_solutions


## 3. Load Markdown, strip frontmatter, chunk

Real pipelines use richer chunking; here we keep it simple so you can **see** retrieval behavior.

In [6]:
def strip_yaml_frontmatter(text: str) -> str:
    text = text.lstrip("\ufeff")
    if not text.startswith("---"):
        return text
    parts = text.split("---", 2)
    if len(parts) >= 3:
        return parts[2].strip()
    return text


def chunk_text(text: str, max_chars: int = 1200, overlap: int = 200):
    text = text.strip()
    chunks = []
    i = 0
    while i < len(text):
        end = min(i + max_chars, len(text))
        if end < len(text):
            para = text.rfind("\n\n", i, end)
            if para != -1 and para > i + max_chars // 2:
                end = para
        piece = text[i:end].strip()
        if len(piece) > 80:
            chunks.append(piece)
        if end >= len(text):
            break
        i = max(i + 1, end - overlap)
    return chunks


def load_corpus(root: Path, max_files: int):
    paths = sorted(root.rglob("*.md"))[:max_files]
    docs = []
    for path in paths:
        try:
            raw = path.read_text(encoding="utf-8", errors="replace")
        except OSError:
            continue
        body = strip_yaml_frontmatter(raw)
        rel = path.relative_to(root)
        for idx, ch in enumerate(chunk_text(body)):
            docs.append(
                {
                    "id": f"{rel.as_posix()}#{idx}",
                    "document": ch,
                    "metadata": {"source": rel.as_posix()},
                }
            )
    return docs


documents = load_corpus(CORPUS_SUBDIR, MAX_FILES)
print(f"Chunks ready: {len(documents)} (from up to {MAX_FILES} .md files)")

Chunks ready: 665 (from up to 100 .md files)


## 4. Embed and store in Chroma (on-disk)

First run downloads the small `all-MiniLM-L6-v2` model (~80MB).

In [8]:
import chromadb
from chromadb.errors import NotFoundError
from chromadb.utils import embedding_functions
from tqdm.auto import tqdm

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

chroma_dir = Path(".").resolve() / f"chroma_{COLLECTION_NAME}"
client = chromadb.PersistentClient(path=str(chroma_dir))

# Reset each full re-index so you do not duplicate rows while experimenting
try:
    client.delete_collection(COLLECTION_NAME)
except NotFoundError:
    pass  # first run for this corpus — nothing to delete yet

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

batch_size = 64
for i in tqdm(range(0, len(documents), batch_size)):
    batch = documents[i : i + batch_size]
    collection.add(
        ids=[d["id"] for d in batch],
        documents=[d["document"] for d in batch],
        metadatas=[d["metadata"] for d in batch],
    )

print("Indexed into:", chroma_dir)

  0%|          | 0/11 [00:00<?, ?it/s]

Indexed into: /Users/mohan/repos/docs-content/learning-rag/chroma_elastic_solutions


## 5. Retrieve: always inspect these passages

Change `QUESTION` and re-run. **If the wrong chunks float to the top, the “RAG answer” will be unreliable** no matter which model you use.

In [10]:
# get-started corpus examples:
# QUESTION = "What is the Elastic Stack and which products are part of it?"
# QUESTION = "What does Kibana do in the Elastic Stack?"

# solutions corpus examples:
QUESTION = "What are the three main Elastic use cases and what should I use for each?"
# QUESTION = "How do I get started with Elastic Observability?"
# QUESTION = "What is semantic search and where do I start in the search solution?"

# out-of-scope test (keep solutions indexed, ask about Fleet install details):
# QUESTION = "How do I install Elastic Agent with Fleet standalone?"

TOP_K = 5

results = collection.query(query_texts=[QUESTION], n_results=TOP_K)

print("QUESTION:", QUESTION)
print("\n--- TOP CHUNKS (read these first) ---\n")
for rank, (doc, meta, dist) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ),
    start=1,
):
    print(f"### {rank}. {meta.get('source', '?')}  (distance: {dist:.4f})")
    print(doc[:900] + ("…" if len(doc) > 900 else ""))
    print()

QUESTION: What are the three main Elastic use cases and what should I use for each?

--- TOP CHUNKS (read these first) ---

### 1. observability/ai/llm-performance-matrix.md  (distance: 0.5665)
for the use case.

Recommended models are those rated **Excellent** or **Great** for the particular use case.
::::

## Proprietary models [_proprietary_models]

Models from third-party LLM providers.

| Provider | Model | **Alert questions** | **APM questions** | **Contextual insights** | **Documentation retrieval** | **Elasticsearch operations** | **{{esql}} generation** | **Execute connector** | **Knowledge retrieval** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Amazon Bedrock | **Claude Sonnet 3.5** | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent | Good | Excellent |
| Amazon Bedrock | **Claude Sonnet 3.7** | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent | Great | Excellent |
| Amazon Bedrock | **Claude Sonnet 4**   | Excel

## 6. Build the “RAG prompt” (context + question)

This is what a downstream model receives in a typical RAG app. **If you do not install Ollama**, copy everything from `BEGIN PROMPT` to `END PROMPT` into ChatGPT (or similar) and compare the reply to the chunks above.

In [11]:
def build_rag_prompt(question: str, retrieved_docs: list[str], sources: list[str]) -> str:
    blocks = []
    for i, (text, src) in enumerate(zip(retrieved_docs, sources), start=1):
        blocks.append(f"[Source {i}: {src}]\n{text}")
    context = "\n\n".join(blocks)
    return (
        "You are assisting a reader of Elastic documentation. "
        "Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. "
        "After each factual claim, add [Source N] matching the source blocks.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    )


sources = [m["source"] for m in results["metadatas"][0]]
prompt = build_rag_prompt(QUESTION, results["documents"][0], sources)
print("--- BEGIN PROMPT (copy from here) ---")
print(prompt)
print("--- END PROMPT ---")

--- BEGIN PROMPT (copy from here) ---
You are assisting a reader of Elastic documentation. Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. After each factual claim, add [Source N] matching the source blocks.

CONTEXT:
[Source 1: observability/ai/llm-performance-matrix.md]
for the use case.

Recommended models are those rated **Excellent** or **Great** for the particular use case.
::::

## Proprietary models [_proprietary_models]

Models from third-party LLM providers.

| Provider | Model | **Alert questions** | **APM questions** | **Contextual insights** | **Documentation retrieval** | **Elasticsearch operations** | **{{esql}} generation** | **Execute connector** | **Knowledge retrieval** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Amazon Bedrock | **Claude Sonnet 3.5** | Excellent | Excellent | Excellent | Excellent | Excellent | Excellent | Good | Excellent |
| Amazon Bedrock | **Claude Sonnet 3.7** | Excellent | Excelle

## 7. (Optional) Local answer with Ollama

1. Install from [ollama.com](https://ollama.com) and start the app (menu bar).
2. In Terminal: `ollama pull llama3.2` (or another small model you prefer).
3. Run the next cell. If you see a connection error, Ollama is not running or the model name is wrong.

This step is **optional**; the learning is in retrieval and the prompt.

In [12]:
import json
import urllib.error
import urllib.request

OLLAMA_MODEL = "llama3.2"

payload = {
    "model": OLLAMA_MODEL,
    "prompt": prompt,
    "stream": False,
}

req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data.get("response", data))
except urllib.error.URLError as e:
    print("Ollama not reachable:", e)
    print("Skip this cell or install/start Ollama and pull", OLLAMA_MODEL)

According to Source 2, Elastic helps you build applications for three main use cases: search, observability, and security.

For each use case, here's a summary of what to use:

1. **Search**: Choose one or both of the following options:
   - Core search capabilities
   - Elasticsearch solution project

2. **Observability**: Use the Observability solution.

3. **Security**: Use the Security solution.

[Source 2: index.md]


## 8. Suggested questions to learn from

After each change to `QUESTION`, re-run sections **5 → 6 → 7**.

### If `CORPUS_FOLDER = "solutions"`

1. **In-corpus:** "What are the three main Elastic use cases?" — expect `solutions/index.md` or similar near the top.
2. **Solution-specific:** "How do I monitor hosts with Elastic Agent?" — watch whether observability quickstarts win over security pages.
3. **Out of scope:** Ask a Fleet install detail while still indexing only `solutions/` — weak retrieval shows corpus boundaries.

### If `CORPUS_FOLDER = "get-started"`

1. **In-corpus:** "What does Kibana do in the Elastic Stack?"
2. **Vague:** "How do I deploy Elastic?"
3. **Out of scope:** Ask something that lives only in `reference/fleet/`.

**Tip:** To compare corpora, switch `CORPUS_FOLDER`, re-run sections **2 → 4**, then ask the **same** question both times.

**For interviews:** be ready to describe chunk boundaries, citation discipline, and what you would change in the docs to make retrieval more reliable—not which embedding model you used.